In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [2]:
train_df=pd.read_csv('train.csv')
test_df=pd.read_csv('test.csv')


In [3]:
train_df.head()

,ID,y,X0,X1,X2,X3,X4,X5,X6,X8,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,0,130.81,k,v,at,a,d,u,j,o,...,0,0,1,0,0,0,0,0,0,0
1,6,88.53,k,t,av,e,d,y,l,o,...,1,0,0,0,0,0,0,0,0,0
2,7,76.26,az,w,n,c,d,x,j,x,...,0,0,0,0,0,0,1,0,0,0
3,9,80.62,az,t,n,f,d,x,l,e,...,0,0,0,0,0,0,0,0,0,0
4,13,78.02,az,v,n,f,d,h,d,n,...,0,0,0,0,0,0,0,0,0,0


In [4]:
test_df.head()

,ID,X0,X1,X2,X3,X4,X5,X6,X8,X10,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,1,az,v,n,f,d,t,a,w,0,...,0,0,0,1,0,0,0,0,0,0
1,2,t,b,ai,a,d,b,g,y,0,...,0,0,1,0,0,0,0,0,0,0
2,3,az,v,as,f,d,a,j,j,0,...,0,0,0,1,0,0,0,0,0,0
3,4,az,l,n,f,d,z,l,n,0,...,0,0,0,1,0,0,0,0,0,0
4,5,w,s,as,c,d,y,i,m,0,...,1,0,0,0,0,0,0,0,0,0


**Following actions should be performed:**

If for any column(s), the variance is equal to zero, then you need to remove those variable(s).

Check for null and unique values for test and train sets.

Apply label encoder.

Perform dimensionality reduction.

Predict your test_df values using XGBoost.

In [5]:
train_var=train_df.var(numeric_only=True)

train_var= train_var.reset_index()
train_var.columns=['Feature','Variance']

train_var.sort_values(by='Variance',ascending=True)

var_train_df=train_var.loc[train_var['Variance']==0,'Feature']
train_df.drop(var_train_df,axis=1,inplace=True)

train_df.drop('ID',axis=1,inplace=True)

In [6]:
test_df.drop(var_train_df,axis=1,inplace=True)
test_df.drop('ID',axis=1,inplace=True)

In [7]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4209 entries, 0 to 4208
Columns: 365 entries, y to X385
dtypes: float64(1), int64(356), object(8)
memory usage: 11.7+ MB


In [8]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4209 entries, 0 to 4208
Columns: 364 entries, X0 to X385
dtypes: int64(356), object(8)
memory usage: 11.7+ MB


In [9]:
##null values check for train set

train_df.isna().sum()

,0
y,0
X0,0
X1,0
X2,0
X3,0
...,...
X380,0
X382,0
X383,0
X384,0


In [10]:
train_cat_col=train_df.select_dtypes(include='object').columns.tolist()


for col in train_cat_col:
  print(train_df[col].unique())
  print(train_df[col].nunique())

['k' 'az' 't' 'al' 'o' 'w' 'j' 'h' 's' 'n' 'ay' 'f' 'x' 'y' 'aj' 'ak' 'am'
 'z' 'q' 'at' 'ap' 'v' 'af' 'a' 'e' 'ai' 'd' 'aq' 'c' 'aa' 'ba' 'as' 'i'
 'r' 'b' 'ax' 'bc' 'u' 'ad' 'au' 'm' 'l' 'aw' 'ao' 'ac' 'g' 'ab']
47
['v' 't' 'w' 'b' 'r' 'l' 's' 'aa' 'c' 'a' 'e' 'h' 'z' 'j' 'o' 'u' 'p' 'n'
 'i' 'y' 'd' 'f' 'm' 'k' 'g' 'q' 'ab']
27
['at' 'av' 'n' 'e' 'as' 'aq' 'r' 'ai' 'ak' 'm' 'a' 'k' 'ae' 's' 'f' 'd'
 'ag' 'ay' 'ac' 'ap' 'g' 'i' 'aw' 'y' 'b' 'ao' 'al' 'h' 'x' 'au' 't' 'an'
 'z' 'ah' 'p' 'am' 'j' 'q' 'af' 'l' 'aa' 'c' 'o' 'ar']
44
['a' 'e' 'c' 'f' 'd' 'b' 'g']
7
['d' 'b' 'c' 'a']
4
['u' 'y' 'x' 'h' 'g' 'f' 'j' 'i' 'd' 'c' 'af' 'ag' 'ab' 'ac' 'ad' 'ae'
 'ah' 'l' 'k' 'n' 'm' 'p' 'q' 's' 'r' 'v' 'w' 'o' 'aa']
29
['j' 'l' 'd' 'h' 'i' 'a' 'g' 'c' 'k' 'e' 'f' 'b']
12
['o' 'x' 'e' 'n' 's' 'a' 'h' 'p' 'm' 'k' 'd' 'i' 'v' 'j' 'b' 'q' 'w' 'g'
 'y' 'l' 'f' 'u' 'r' 't' 'c']
25


In [11]:
## null values check for test sets

test_df.isna().sum()

,0
X0,0
X1,0
X2,0
X3,0
X4,0
...,...
X380,0
X382,0
X383,0
X384,0


In [12]:
test_cat_col=test_df.select_dtypes(include='object').columns.tolist()

for col in test_cat_col:
  print(test_df[col].unique())
  print(test_df[col].nunique())

['az' 't' 'w' 'y' 'x' 'f' 'ap' 'o' 'ay' 'al' 'h' 'z' 'aj' 'd' 'v' 'ak'
 'ba' 'n' 'j' 's' 'af' 'ax' 'at' 'aq' 'av' 'm' 'k' 'a' 'e' 'ai' 'i' 'ag'
 'b' 'am' 'aw' 'as' 'r' 'ao' 'u' 'l' 'c' 'ad' 'au' 'bc' 'g' 'an' 'ae' 'p'
 'bb']
49
['v' 'b' 'l' 's' 'aa' 'r' 'a' 'i' 'p' 'c' 'o' 'm' 'z' 'e' 'h' 'w' 'g' 'k'
 'y' 't' 'u' 'd' 'j' 'q' 'n' 'f' 'ab']
27
['n' 'ai' 'as' 'ae' 's' 'b' 'e' 'ak' 'm' 'a' 'aq' 'ag' 'r' 'k' 'aj' 'ay'
 'ao' 'an' 'ac' 'af' 'ax' 'h' 'i' 'f' 'ap' 'p' 'au' 't' 'z' 'y' 'aw' 'd'
 'at' 'g' 'am' 'j' 'x' 'ab' 'w' 'q' 'ah' 'ad' 'al' 'av' 'u']
45
['f' 'a' 'c' 'e' 'd' 'g' 'b']
7
['d' 'b' 'a' 'c']
4
['t' 'b' 'a' 'z' 'y' 'x' 'h' 'g' 'f' 'j' 'i' 'd' 'c' 'af' 'ag' 'ab' 'ac'
 'ad' 'ae' 'ah' 'l' 'k' 'n' 'm' 'p' 'q' 's' 'r' 'v' 'w' 'o' 'aa']
32
['a' 'g' 'j' 'l' 'i' 'd' 'f' 'h' 'c' 'k' 'e' 'b']
12
['w' 'y' 'j' 'n' 'm' 's' 'a' 'v' 'r' 'o' 't' 'h' 'c' 'k' 'p' 'u' 'd' 'g'
 'b' 'q' 'e' 'l' 'f' 'i' 'x']
25


In [15]:
from sklearn.preprocessing import OrdinalEncoder

ordinal=OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)

train_df[train_cat_col]=ordinal.fit_transform(train_df[train_cat_col])
test_df[train_cat_col]=ordinal.transform(test_df[train_cat_col])

In [16]:
train_df.head()

,y,X0,X1,X2,X3,X4,X5,X6,X8,X10,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,130.81,32.0,23.0,17.0,0.0,3.0,24.0,9.0,14.0,0,...,0,0,1,0,0,0,0,0,0,0
1,88.53,32.0,21.0,19.0,4.0,3.0,28.0,11.0,14.0,0,...,1,0,0,0,0,0,0,0,0,0
2,76.26,20.0,24.0,34.0,2.0,3.0,27.0,9.0,23.0,0,...,0,0,0,0,0,0,1,0,0,0
3,80.62,20.0,21.0,34.0,5.0,3.0,27.0,11.0,4.0,0,...,0,0,0,0,0,0,0,0,0,0
4,78.02,20.0,23.0,34.0,5.0,3.0,12.0,3.0,13.0,0,...,0,0,0,0,0,0,0,0,0,0


In [17]:
test_df.head()

,X0,X1,X2,X3,X4,X5,X6,X8,X10,X12,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,20.0,23.0,34.0,5.0,3.0,-1.0,0.0,22.0,0,0,...,0,0,0,1,0,0,0,0,0,0
1,40.0,3.0,7.0,0.0,3.0,-1.0,6.0,24.0,0,0,...,0,0,1,0,0,0,0,0,0,0
2,20.0,23.0,16.0,5.0,3.0,-1.0,9.0,9.0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,20.0,13.0,34.0,5.0,3.0,-1.0,11.0,13.0,0,0,...,0,0,0,1,0,0,0,0,0,0
4,43.0,20.0,16.0,2.0,3.0,28.0,8.0,12.0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [18]:
X_train=train_df.drop('y',axis=1)
y_train=train_df['y']

In [20]:
test_df=test_df[X_train.columns]

In [21]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_scaled_train=scaler.fit_transform(X_train)

X_scaled_test=scaler.transform(test_df)

In [22]:
from sklearn.decomposition import PCA

pca=PCA(n_components=0.90)

In [23]:
train_pca_values=pca.fit_transform(X_scaled_train)

train_pca=pd.DataFrame(train_pca_values)

In [24]:
test_pca_values=pca.transform(X_scaled_test)

test_pca=pd.DataFrame(test_pca_values)

In [25]:
cumm_sum=np.sum(pca.explained_variance_ratio_)
print(cumm_sum)

0.9005330510036982


In [26]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score

xgb=XGBRegressor()

scores=cross_val_score(xgb,train_pca,y_train,cv=5,scoring='r2')

print(scores)

print(scores.mean())


[0.40488684 0.35018708 0.4242218  0.36018213 0.48791394]
0.4054783575614144


In [27]:
xgb.fit(train_pca,y_train)

y_pred_test=xgb.predict(test_pca)

In [28]:
test_df['y']=y_pred_test